# 1000 Soils – Random Forest

This notebook builds **regression** and **classification** Random Forest models to predict soil respiration (`respiration_ppm_co2_c`) from biogeochemical and site metadata features collected across the 1000 Soils dataset.

**Workflow overview:**
1. Download and merge biogeochemical and metadata tables from Zenodo
2. Label-encode categorical site descriptors for use as numeric features
3. Select features and remove rows with missing values
4. Train a Random Forest **regressor** and evaluate with R² and RMSE
5. Bin the target into Low / Medium / High tertiles and train a Random Forest **classifier**

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
!pip install zenodo-get

In [ ]:
# Download data from Zenodo
cmd = ["zenodo_get", "10.5281/zenodo.15328215"]
working_directory = "1000Soil_data"
os.makedirs(working_directory, exist_ok=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=working_directory)
output, error = process.communicate()

if error:
    print("An error occurred:", error.decode())
else:
    print("Download complete!")
    print(output.decode())


## Data Loading & Preprocessing

Loads two data sources and merges them on `Site`:
- **`1000S_processed_BGC_summary.csv`** — biogeochemical measurements (pH, C/N, moisture, etc.) per soil core section.
- **`1000Soils_Metadata_Site_Mastersheet_v1.xlsx`** — site-level metadata (coordinates, land cover, etc.).

After merging, missing coordinates for a small number of sites (WSU, OAES, TEAK) are filled in manually, and each site is assigned a `BiomeType` label from a hand-coded lookup dictionary.

In [ ]:
# Load metadata
metadata = pd.read_excel("1000Soil_data/1000Soils_Metadata_Site_Mastersheet_v1.xlsx")
metadata['Site'] = metadata['Site_Code'].str.replace(' ', '')

# Load biogeochemical data
df_BG = pd.read_csv("1000Soil_data/1000S_processed_BGC_summary.csv")
print(f"BGC rows: {len(df_BG)}")

# Extract site code and create unique sample identifier
df_BG.insert(1, 'Site', df_BG['Site_ID'].str.split('_').str[1])
df_BG['Sample'] = df_BG['Site'] + "_" + df_BG['Core_Section']

# Merge with metadata
df_BG = df_BG.merge(metadata, how='outer', on='Site')

# Fix missing coordinates
wsu_rows  = df_BG['Site'].str.contains("WSU", case=False, na=False)
df_BG.loc[wsu_rows,                   ['Lat', 'Long']] = [46.2523,    -119.737]
df_BG.loc[df_BG['Site'].isin(['OAES']), ['Lat', 'Long']] = [35.410599,  -99.058779]
df_BG.loc[df_BG['Site'].isin(['TEAK']), ['Lat', 'Long']] = [37.00583,  -119.00602]

# Biome mapping
biome_dict = {
    'ANZA': 'Desert shrubland',   'FTA3': 'Desert shrubland',
    'FTA5': 'Desert shrubland',   'JORN': 'Desert shrubland',
    'MOAB': 'Desert shrubland',   'OCTB': 'Desert shrubland',
    'OCTU': 'Desert shrubland',   'ONAQ': 'Desert shrubland',
    'PRS2': 'Desert shrubland',   'SRER': 'Desert shrubland',
    'SRR1': 'Desert shrubland',   'UT32': 'Desert shrubland',
    'WSU1': 'Desert shrubland',   'WSU2': 'Desert shrubland',
    'WSU3': 'Desert shrubland',   'OAES': 'Desert shrubland',
    'SJER': 'Mediterranean woodlands',
    'BLAN': 'Temperate forests',  'CFS2': 'Temperate forests',
    'DELA': 'Temperate forests',  'GRSM': 'Temperate forests',
    'LENO': 'Temperate forests',  'MLBS': 'Temperate forests',
    'NWBA': 'Temperate forests',  'NWBB': 'Temperate forests',
    'NWBC': 'Temperate forests',  'ORNL': 'Temperate forests',
    'PPRH': 'Temperate forests',  'PRS1': 'Temperate forests',
    'SCBI': 'Temperate forests',  'SERC': 'Temperate forests',
    'TALL': 'Temperate forests',  'WLLO': 'Temperate forests',
    'WLUP': 'Temperate forests',
    'CFS1': 'Temperate coniferous forests', 'DSNY': 'Temperate coniferous forests',
    'JERC': 'Temperate coniferous forests', 'OKPF': 'Temperate coniferous forests',
    'OSBS': 'Temperate coniferous forests', 'PETF': 'Temperate coniferous forests',
    'PHTU': 'Temperate coniferous forests', 'RMNP': 'Temperate coniferous forests',
    'SOAP': 'Temperate coniferous forests', 'UT12': 'Temperate coniferous forests',
    'UT19': 'Temperate coniferous forests', 'UT23': 'Temperate coniferous forests',
    'UT24': 'Temperate coniferous forests', 'WY01': 'Temperate coniferous forests',
    'WY03': 'Temperate coniferous forests', 'WY09': 'Temperate coniferous forests',
    'WY10': 'Temperate coniferous forests', 'WY15': 'Temperate coniferous forests',
    'TEAK': 'Temperate coniferous forests',
    'CLBJ': 'Temperate grasslands', 'CPER': 'Temperate grasslands',
    'DCFS': 'Temperate grasslands', 'ISCC': 'Temperate grasslands',
    'ISNC': 'Temperate grasslands', 'KONA': 'Temperate grasslands',
    'KONZ': 'Temperate grasslands', 'NOGP': 'Temperate grasslands',
    'UKFS': 'Temperate grasslands', 'STER': 'Temperate grasslands',
}

df_BG['BiomeType'] = df_BG['Site'].map(biome_dict)
df_BG['Sample'] = df_BG['Site'] + "_" + df_BG['Core_Section']
df_BG = df_BG.rename(columns={'Site_ID': 'Sample_ID'})

df_merged = df_BG
print(f"Merged rows: {len(df_merged)}")
df_merged.head(5)


### Categorical Encoding

Random Forest requires numeric inputs. Four categorical columns are label-encoded into integer `*_enc` columns using `sklearn.LabelEncoder`:
- **`BiomeType_enc`** — broad ecosystem type (e.g. Temperate forest, Desert shrubland).
- **`Vegetation_enc`** — local vegetation description.
- **`Weather_enc`** — weather conditions at time of sample collection.
- **`Site_enc`** — individual site code.

Rows where the original column is `NaN` receive code `-1` so they are distinguishable from valid categories. The original string columns remain in the dataframe but are excluded from modelling via `drop_cols`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical columns for use as numeric features in Random Forest
cat_cols = ['BiomeType', 'Weather', 'Vegetation', 'Site']

le = LabelEncoder()
for col in cat_cols:
    enc_col = f'{col}_enc'
    # Fit only on non-null values; NaN rows get -1
    non_null = df_merged[col].notna()
    df_merged[enc_col] = -1
    df_merged.loc[non_null, enc_col] = le.fit_transform(df_merged.loc[non_null, col])

print("Encoded columns added:")
for col in cat_cols:
    n_classes = df_merged[f'{col}_enc'].nunique()
    print(f"  {col}_enc : {n_classes} unique codes  (sample: {df_merged[f'{col}_enc'].value_counts().head(3).to_dict()})")


### Define Target & Filter Rows

Sets `respiration_ppm_co2_c` as the prediction target and creates `df_target` by dropping all rows where this column is `NaN`. Only these rows are used for feature selection and model training, since a sample with no known respiration value cannot contribute to supervised learning. The filtered dataframe is also saved as `target_rows.csv` for inspection.

In [ ]:
target = 'respiration_ppm_co2_c'

# Work on rows where the target is available
df_target = df_merged.dropna(subset=[target])
df_target.to_csv("target_rows.csv", index=False)
n_target = len(df_target)
print(f"Rows with target non-null: {n_target}")


### Missing Data Inspection

Summarises missingness across **all** columns in `df_target`. Useful for understanding which features have sparse coverage before committing to the 30% threshold filter in the next step. Columns with zero missing values are excluded from the display to keep it concise.

In [ ]:
# Missing data summary for all columns in df_target
missing_summary = pd.DataFrame({
    'Missing Count': df_target.isna().sum(),
    'Missing %': (df_target.isna().mean() * 100).round(1)
}).sort_values('Missing %', ascending=False)
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

print(f"Total rows (target non-null): {n_target}")
print(f"Columns with any missing data: {len(missing_summary)}\n")
display(missing_summary)


## Feature Selection

Selects numeric feature columns from `df_target` (rows where the target is non-null) by:
1. Excluding identifier columns, free-text fields, and variables that would directly leak the target (other respiration measurements).
2. Excluding encoded categoricals with low expected predictive value (`Vegetation_enc`, `Weather_enc`, `Site_enc`), while retaining `BiomeType_enc`.
3. Applying a **≤ 30% missing data threshold** — any candidate feature with more than 30% NaN values across target rows is dropped.
4. Calling `.dropna()` on the final feature + target subset to ensure a complete-case dataset with no remaining NaN values.

The resulting `X` (feature matrix) and `y` (target vector) are used by all downstream model cells.

NOTE: For the sake of demonstration, we are dropping the encoded Vegetation, Weather, and Site columns. We are keeping the BiomeType, which was generated in an earilier code cell.

In [ ]:
# Identifier / non-feature columns to always exclude
# Note: BiomeType, Weather, Vegetation, Site are encoded as *_enc numeric columns above
drop_cols = [
    'Sample', 'Site_Code', 'Site_Name_Long', 'Core_collector',
    'DateTime_Collected (24 Hr format)',
    'ROI.volume.xyz', 'Location',
    'respiration_mg_co2_c_g_soil_day_96hour',
    'respiration_mg_co2_c_g_soil_day_24hour',
    'VolWaterContent.m3/m3',
    'Vegetation_enc', # Dropped for demo
    'Weather_enc', # Dropped for demo
    'Site_enc', # Dropped for demo
]

# Candidate numeric feature columns
candidate_cols = [
    c for c in df_target.columns
    if c != target
    and c not in drop_cols
    and pd.api.types.is_numeric_dtype(df_target[c])
]

# Keep only columns with <=30% missing among target rows
missing_pct = df_target[candidate_cols].isna().mean()
feature_cols = missing_pct[missing_pct <= 0.30].index.tolist()

dropped = [c for c in candidate_cols if c not in feature_cols]
print(f"\nFeature columns kept  : {len(feature_cols)}")
print(f"Feature columns dropped (>30% missing): {len(dropped)}")
if dropped:
    print("  Dropped:", dropped)

df_ml = df_target[[target] + feature_cols].copy().dropna() # The dropna() here ensures we only work with rows that have complete data for the target and selected features.


### Assemble Feature Matrix and Target Vector

Extracts `X` (feature matrix) and `y` (target vector) from `df_ml` — the complete-case subset produced by the feature selection step. These objects are reused by all downstream modelling cells (regression, classification) so the same rows and features are used consistently throughout the notebook.

In [ ]:
# Build X and y
X = df_ml[feature_cols]
y = df_ml[target]
print(f"\nSamples : {len(df_ml)}")
print(f"Features: {len(feature_cols)}")

## Random Forest Regression

Trains a `RandomForestRegressor` on an 80/20 train/test split and reports **R²** and **RMSE** on the held-out test set.

Key hyperparameters:
- `n_estimators=500` — 500 trees for stable importance estimates.
- `max_features='sqrt'` — each split considers √(n_features) candidates, reducing correlation between trees.
- `min_samples_leaf=2` — prevents overfitting by requiring at least 2 samples at every leaf.
- `bootstrap=True` — each tree is fit on a bootstrap sample of the training data.

Outputs:
- **Predicted vs Actual scatter plot** with a 1:1 reference line.
- **Horizontal bar chart** of the top 20 features ranked by mean decrease in impurity (MDI).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Same train/test split as XGBoost (same random_state)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Fit Random Forest
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_rf, y_train_rf)

# Evaluate
y_pred_rf = rf.predict(X_test_rf)
r2_rf   = r2_score(y_test_rf, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
print(f"Random Forest  Test R²  : {r2_rf:.3f}")
print(f"Random Forest  Test RMSE: {rmse_rf:.4f}")

# ── Predicted vs Actual ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_test_rf, y_pred_rf, alpha=0.7, edgecolors='k', linewidths=0.4)
lims = [min(y_test_rf.min(), y_pred_rf.min()), max(y_test_rf.max(), y_pred_rf.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='1:1 line')
ax.set_xlabel('Actual respiration (mg CO₂-C g⁻¹ soil day⁻¹)')
ax.set_ylabel('Predicted')
ax.set_title(f'Random Forest – Predicted vs Actual\nR² = {r2_rf:.3f}  RMSE = {rmse_rf:.4f}')
ax.legend()

# ── Feature Importances (top 20) ────────────────────────────────────────────
rf_importances = pd.Series(rf.feature_importances_, index=feature_cols)
rf_top20 = rf_importances.nlargest(20).sort_values()

ax2 = axes[1]
rf_top20.plot(kind='barh', ax=ax2, color='steelblue', edgecolor='k', linewidth=0.4)
ax2.set_xlabel('Feature Importance (mean decrease impurity)')
ax2.set_title('Top 20 Feature Importances – Random Forest')

plt.tight_layout()
plt.show()


## Random Forest Classification

Converts the continuous respiration target into an ordinal classification problem with three equal-frequency classes using **tertile (quantile) cuts**:
- **Low** — bottom third of respiration values
- **Medium** — middle third
- **High** — top third

Using `pd.qcut` (quantile-based) ensures each class has roughly the same number of samples, avoiding class imbalance.

Two sub-steps follow:
1. **Binning** — create `y_class` and inspect the class distribution and cut-point values.
2. **Training & evaluation** — fit a `RandomForestClassifier` with stratified splits and balanced class weights, then report accuracy, precision, recall, F1, and a confusion matrix.

### Bin Target into Classes

Uses `pd.qcut` to divide `y` into three equal-frequency tertiles labelled **Low**, **Medium**, and **High**. The resulting `y_class` is used as the classification target. Printing the class counts and cut-points confirms the binning is balanced and shows the respiration thresholds that separate each class.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
)

# ── Create class labels from tertile cuts ────────────────────────────────────
class_labels = ['Low', 'Medium', 'High']
y_class = pd.qcut(y, q=3, labels=class_labels)

print("Class distribution (tertile cuts):")
print(y_class.value_counts().sort_index())
print(f"\nCut points: {pd.qcut(y, q=3, retbins=True)[1].round(4)}")


### Train & Evaluate Classifier

Trains a `RandomForestClassifier` on a stratified 80/20 split (stratification preserves the Low/Medium/High class ratio in both train and test sets).

Key hyperparameters:
- `class_weight='balanced'` — adjusts tree split criteria to compensate for any residual class imbalance after tertile binning.
- `min_samples_leaf=2` — avoids overfitting to single noisy samples.

Evaluation outputs:
- **Accuracy** — fraction of test samples correctly classified.
- **Classification report** — per-class precision, recall, and F1-score.
- **Confusion matrix** — visualises which classes are confused with each other; off-diagonal cells indicate misclassifications.
- **Feature importance bar chart** — MDI-based importance for the top 20 features (purple bars for visual distinction from the regression plot).

In [ ]:
# ── Train / test split ───────────────────────────────────────────────────────
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class)

# ── Fit classifier ───────────────────────────────────────────────────────────
rfc = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=2,
    class_weight='balanced',
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)
rfc.fit(X_train_clf, y_train_clf)

# ── Evaluate ─────────────────────────────────────────────────────────────────
y_pred_clf = rfc.predict(X_test_clf)
acc = accuracy_score(y_test_clf, y_pred_clf)
print(f"Accuracy: {acc:.3f}\n")
print(classification_report(y_test_clf, y_pred_clf, target_names=class_labels))

# ── Confusion matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test_clf, y_pred_clf, labels=class_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nAccuracy = {acc:.3f}')

# ── Feature importances (top 20) ─────────────────────────────────────────────
rfc_importances = pd.Series(rfc.feature_importances_, index=feature_cols)
rfc_top20 = rfc_importances.nlargest(20).sort_values()

rfc_top20.plot(kind='barh', ax=axes[1], color='mediumpurple', edgecolor='k', linewidth=0.4)
axes[1].set_xlabel('Feature Importance (mean decrease impurity)')
axes[1].set_title('Top 20 Feature Importances – RF Classifier')

plt.tight_layout()
plt.show()
